# DevContext AI - Full Agent Capstone Checkpoint (Ingestion)

## Problem Statement
Build a production-grade ingestion pipeline for the `tiangolo/fastapi` GitHub repository — ingest Python source files, English markdown docs, and a SQL schema into three separate ChromaDB collections using source-appropriate chunking and embedding models.

## My Approach

Split ingestion into three source types with different strategies:

- **Python** → AST-aware chunking (function and class method level), embedded with `BAAI/bge-base-en-v1.5` (code-optimized)
- **Docs** → Recursive character splitter (chunk_size=900, overlap=150), embedded with `all-mpnet-base-v2` (semantic-optimized)
- **SQL** → Statement + column level chunking via `sqlglot`, embedded with MPNet

Used HuggingFace Inference API (free tier) instead of local models due to PC limitations. Added retry logic for 503s and timeouts. Parallelized sub-batches across 3 workers with `ThreadPoolExecutor` to reduce total wall time. Scoped docs to `docs/en` only to avoid ingesting all language translations.

## What I Learned

- **Chunk size has a compounding effect.** At chunk_size=300, a single large docs folder generates thousands of chunks → hundreds of HF API calls → timeout spiral. Raising to 900 cut API calls by ~3x with minimal retrieval quality loss.
- **Parallelism is the right lever, not batch size.** Increasing batch size from 50 to 200 didn't help because the bottleneck was HF model cold-start latency, not payload size. 3 parallel workers solved the actual problem.
- **Scope your data early.** Ingesting all language docs multiplied chunk count unnecessarily. Filtering to `docs/en` only removed noise and cut volume significantly.

## Where I Got Stuck

This is where the sprint actually happened.

My first attempt: chunk_size=300, no parallelism, all language docs included, single-threaded HF API calls. Doc ingestion started and kept running - 30+ minutes, still going. I increased batch size from 50 to 200 thinking it would reduce round trips. It didn't move the needle. Still hitting HF rate limits and model cold-start timeouts repeatedly.

I didn't immediately understand *why* batch size wasn't helping. The real issue was that I was sending one batch at a time and waiting for each response sequentially - so HF cold-start latency (15-30s per 503) was multiplying across every sub-batch. Bigger batches just meant fewer, slower calls - not faster ingestion.

The fix came from rethinking the bottleneck: switch from sequential to parallel sub-batches (`ThreadPoolExecutor`), raise chunk_size to 900 to reduce total chunk count, and scope docs to English only. Combined, these brought total ingestion time from 30+ minutes (and still running) down to ~5 minutes for all three collections.

## What I'd Do Differently

- **Start with a dry run.** Print chunk counts and estimated API calls before touching the HF API - the notebook now does this. I'd add this as step 1 on any future ingestion pipeline.
- **Profile before optimizing.** I spent time on batch size when the bottleneck was parallelism. A simple `time.time()` around each sub-batch call would have shown this in 2 minutes.
- **Use local models for development.** HF Inference API is the right production-frugal choice, but for iteration speed, running `sentence-transformers` locally with even a small model would have tightened the feedback loop significantly.
- The ingestion blocklist is not yet wired — that goes in the agent notebook. Worth noting that `*.env` and credential patterns should be checked here at walk time, not at query time.

## ingestion of fast api

In [4]:
import os
import ast
import requests
import numpy as np
import chromadb
import chromadb.utils.embedding_functions as embedding_functions
import concurrent.futures
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sqlglot import parse, exp
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv("HUGGING_FACE_API_KEY")

MODEL_IDS = {
    "MPNet_doc_sql": "sentence-transformers/all-mpnet-base-v2",
    "baai_code":     "BAAI/bge-base-en-v1.5"
}


d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Chunking 
# chunk_size=900: ~3x fewer chunks than 300 → ~3x fewer API calls
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)

def ast_chunking(code: str) -> list:
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return []
    chunks = []
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, ast.FunctionDef):
            seg = ast.get_source_segment(code, node)
            if seg:
                chunks.append(seg)
        elif isinstance(node, ast.ClassDef):
            for child in ast.iter_child_nodes(node):
                if isinstance(child, ast.FunctionDef):
                    seg = ast.get_source_segment(code, child)
                    if seg:
                        chunks.append(seg)
    return chunks

def chunk_sql_schema(sql: str) -> list:
    parsed = parse(sql)
    chunks = []
    for stmt in parsed:
        chunks.append(stmt.sql(pretty=True))
        schema = stmt.find(exp.Schema)
        if schema:
            for col in schema.expressions:
                chunks.append(col.sql(pretty=True))
    return chunks


In [9]:
#Embedding with retry 
def get_api_embeddings(texts: list, model_id: str) -> list | None:
    if not texts:
        return []
    api_url = f"https://router.huggingface.co/hf-inference/models/{model_id}/pipeline/feature-extraction"
    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    payload = {"inputs": texts, "options": {"wait_for_model": True}}

    for attempt in range(3):
        try:
            response = requests.post(api_url, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                arr = np.array(response.json())
                if arr.ndim == 3:
                    arr = arr.mean(axis=1)
                return arr.tolist()
            elif response.status_code == 503:
                wait = 15 * (attempt + 1)
                print(f" HF model loading — retrying in {wait}s (attempt {attempt+1}/3)...")
                time.sleep(wait)
            else:
                print(f"HTTP {response.status_code}: {response.text[:300]}")
                return None
        except requests.exceptions.Timeout:
            print(f"Timeout on attempt {attempt+1} — retrying...")
            time.sleep(5)
        except Exception as e:
            print(f"Unexpected error: {e}")
            return None
    print("All 3 attempts failed — sub-batch dropped")
    return None


In [10]:

#  ChromaDB setup 
cwd = os.getcwd()
db_path = os.path.abspath(os.path.join(cwd, "..", "chroma_db"))

client = chromadb.PersistentClient(path=db_path)

for name in ["docbase_fastapi", "schemabase_fastapi", "codebase_fastapi"]:
    try:
        client.delete_collection(name)
        print(f"🗑  Dropped old collection: {name}")
    except Exception:
        pass

hf_ef_mpnet = embedding_functions.HuggingFaceEmbeddingFunction(
    api_key=HF_TOKEN, model_name=MODEL_IDS["MPNet_doc_sql"]
)
hf_ef_bge = embedding_functions.HuggingFaceEmbeddingFunction(
    api_key=HF_TOKEN, model_name=MODEL_IDS["baai_code"]
)

collections = {
    "doc":    client.get_or_create_collection("docbase_fastapi",    embedding_function=hf_ef_mpnet),
    "schema": client.get_or_create_collection("schemabase_fastapi", embedding_function=hf_ef_mpnet),
    "code":   client.get_or_create_collection("codebase_fastapi",   embedding_function=hf_ef_bge),
}


🗑  Dropped old collection: docbase_fastapi
🗑  Dropped old collection: schemabase_fastapi
🗑  Dropped old collection: codebase_fastapi


In [11]:
# ── Flush function — parallel sub-batches ────────────────────
SUB_BATCH  = 64   # HF free tier sweet spot
MAX_WORKERS = 3   # 3 parallel threads → 3x throughput without hammering rate limit

def flush_buffer(category: str, documents: list, ids: list, metadatas: list):
    if not ids:
        return
    model_id = MODEL_IDS["baai_code"] if category == "code" else MODEL_IDS["MPNet_doc_sql"]

    sub_batches = [
        (documents[i:i+SUB_BATCH], ids[i:i+SUB_BATCH], metadatas[i:i+SUB_BATCH])
        for i in range(0, len(ids), SUB_BATCH)
    ]
    print(f"\n '{category}': {len(ids)} chunks → {len(sub_batches)} sub-batches X {MAX_WORKERS} parallel workers")

    def embed_and_store(batch):
        b_docs, b_ids, b_metas = batch
        embeddings = get_api_embeddings(b_docs, model_id)
        if embeddings:
            collections[category].add(
                embeddings=embeddings,
                documents=b_docs,
                ids=b_ids,
                metadatas=b_metas
            )
            return len(b_ids)
        print(f"Sub-batch of {len(b_ids)} dropped in '{category}'")
        return 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(executor.map(embed_and_store, sub_batches))

    print(f"'{category}': {sum(results)}/{len(ids)} stored")

# ── Repository walk — collect ALL chunks first ───────────────
repo_path   = "../data/fastapi"
skip_folders = {'.github', 'docs_src', 'tests', 'scripts', 'fastapi-slim', 'site', 'build', 'dist'}

all_chunks = {
    "doc":    {"documents": [], "ids": [], "metadatas": []},
    "schema": {"documents": [], "ids": [], "metadatas": []},
    "code":   {"documents": [], "ids": [], "metadatas": []},
}

print("Scanning repository...")

for root, dirnames, filenames in os.walk(repo_path):
    dirnames[:] = [d for d in dirnames if d not in skip_folders]

    # Keep docs/en only - prune all other language folders
    path_parts = root.replace(os.sep, "/").split("/")
    if "docs" in path_parts:
        if path_parts[-1] == "docs":
            dirnames[:] = [d for d in dirnames if d == "en"]
        elif "en" not in path_parts:
            dirnames[:] = []
            continue

    for file in filenames:
        lower_file = file.lower()
        if not lower_file.endswith(('.md', '.rst', '.txt', '.sql', '.py')):
            continue

        file_path = os.path.join(root, file)
        rel_path  = os.path.relpath(file_path, repo_path)

        # Extra guard: skip non-English docs that slipped through
        if "docs" in rel_path and f"docs{os.sep}en" not in rel_path:
            continue

        try:
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                raw = f.read().strip()
        except Exception:
            continue
        if not raw:
            continue

        if lower_file.endswith(('.md', '.rst', '.txt')):
            category = "doc"
            chunks   = doc_splitter.split_text(raw)
        elif lower_file.endswith('.sql'):
            category = "schema"
            chunks   = chunk_sql_schema(raw)
        elif lower_file.endswith('.py'):
            category = "code"
            chunks   = ast_chunking(raw)
        else:
            continue

        if not chunks:
            continue

        for idx, chunk in enumerate(chunks):
            chunk_id = f"{category}_{rel_path.replace(os.sep, '_')}_chk{idx}"
            all_chunks[category]["documents"].append(chunk)
            all_chunks[category]["ids"].append(chunk_id)
            all_chunks[category]["metadatas"].append({
                "file_name":   file,
                "rel_path":    rel_path,
                "chunk_index": idx
            })

# Print scale before committing API calls 
print("\nChunk counts before embedding:")
for cat, buf in all_chunks.items():
    sub_batch_count = -(-len(buf["ids"]) // SUB_BATCH)  # ceiling division
    print(f"   {cat:6s}: {len(buf['ids']):>5} chunks → ~{sub_batch_count} HF API calls")

total_calls = sum(-(-len(buf["ids"]) // SUB_BATCH) for buf in all_chunks.values())
print(f"\n   Total estimated HF API calls: {total_calls}")
print("   Proceeding with parallel ingestion...\n")

#Flush all categories 
for category in ["doc", "schema", "code"]:
    buf = all_chunks[category]
    flush_buffer(category, buf["documents"], buf["ids"], buf["metadatas"])

#  Final verification 
print("\n=== Ingestion Complete ===")
for name, coll in collections.items():
    print(f"  Collection '{coll.name}': {coll.count()} items")

Scanning repository...

Chunk counts before embedding:
   doc   :  1388 chunks → ~22 HF API calls
   schema:    16 chunks → ~1 HF API calls
   code  :   222 chunks → ~4 HF API calls

   Total estimated HF API calls: 27
   Proceeding with parallel ingestion...


 'doc': 1388 chunks → 22 sub-batches X 3 parallel workers
Timeout on attempt 1 — retrying...
Timeout on attempt 2 — retrying...
'doc': 1388/1388 stored

 'schema': 16 chunks → 1 sub-batches X 3 parallel workers
'schema': 16/16 stored

 'code': 222 chunks → 4 sub-batches X 3 parallel workers
'code': 222/222 stored

=== Ingestion Complete ===
  Collection 'docbase_fastapi': 1388 items
  Collection 'schemabase_fastapi': 16 items
  Collection 'codebase_fastapi': 222 items
